In [1]:
import pandas as pd
from groq import Groq
import os

with open('groq.txt') as file:
    api_key = str(file.readline())

os.environ['GROQ_API_KEY'] = api_key

# Load the fintech dataset
df = pd.read_csv("fintechdf_categorized.csv", encoding="latin-1")

# Get dataset info for context
columns_info = df.dtypes.to_string()
sample_data = df.head(3).to_string()
shape_info = f"Dataset has {df.shape[0]} rows and {df.shape[1]} columns"

# Create context about the dataset
data_context = f"""
You have access to a fintech startup dataset with the following structure:

{shape_info}

Columns and their types:
{columns_info}

Sample data (first 3 rows):
{sample_data}

The dataset contains information about European fintech startups including:
- Basic info: ID, Name, Long description, HQ city, Launch year, Founders
- Funding: Total funding (EUR M), Last round
- Category flags (binary 0/1): digital_payments_processing, insurance_insurtech_underwriting, 
  blockchain_cryptocurrency_assets, lending_credit_financing, investment_wealth_trading,
  corporate_treasury_accounting, sustainability_carbon_climate, tax_legal_planning, mergers_acquisitions_advisory

When asked a question, generate Python pandas code that operates on the dataframe 'df' to answer it.
Return ONLY the executable Python code, no explanations. The code should end with a print statement showing the result.
"""

# User question - change this to ask different questions
user_prompt = "What is the average amount of funding raised by companies that belong to the category digital_payments_processing?"

In [2]:
# Initialize Groq client
client = Groq()

# Get the LLM to generate code to answer the question
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": data_context
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    temperature=0.1,
    max_tokens=1024,
)

# Get the generated code
generated_code = completion.choices[0].message.content
print(generated_code)


```python
import pandas as pd

average_funding = df[df['digital_payments_processing'] == 1]['Total funding (EUR M)'].mean()
print(average_funding)
```


In [4]:

# Clean the generated code (remove markdown code blocks if present)
code_to_run = generated_code
if "```python" in code_to_run:
    code_to_run = code_to_run.split("```python")[1].split("```")[0]
elif "```" in code_to_run:
    code_to_run = code_to_run.split("```")[1].split("```")[0]

# Execute the code
exec(code_to_run)

42.49473684210526
